
# people_segment.csv → persona_attributes_weighted.jsonl

이 노트북은 **인구통계 vs 구매성향(세그먼트) 비중을 조정**하여,  
각 인물(행)마다 **속성별 가중치 합=1.0**이 되도록 정규화한 JSONL을 생성합니다.

- 입력: `people_segment.csv`
- 출력: `persona_attributes_weighted.jsonl`, `persona_attributes_weighted_preview.json`
- 비중 조정: `DEMO_SHARE`, `BEHAV_SHARE` (합=1.0 권장)
- 추가 조정: `DEMO_FIELD_WEIGHTS`, `BEHAV_FIELD_MULTIPLIERS`


In [ ]:

# =============================
# 0) CONFIG — 조정 가능한 설정
# =============================
from pathlib import Path

# 입력/출력 경로
INPUT_CSV = Path("/mnt/data/people_segment.csv")
OUT_JSONL = Path("/mnt/data/persona_attributes_weighted.jsonl")
OUT_PREVIEW = Path("/mnt/data/persona_attributes_weighted_preview.json")

# 인구통계 vs 성향 전체 비중 (합=1.0 권장)
DEMO_SHARE = 0.30   # 인구통계 전체 비중 (예: 0.20~0.40)
BEHAV_SHARE = 0.70  # 성향/세그먼트 전체 비중 (예: 0.60~0.80)

# 인구통계 내부 항목별 가중 (없으면 균등 분배)
# - 키는 CSV 컬럼명(SQ3, HQ2...)을 사용
DEMO_FIELD_WEIGHTS = {
    # 예: "HQ2": 2.0, "SQ3": 1.0   # 연령 더 중요
}

# 성향 내부 항목별 multiplier (정규화 전 강조/약화)
# - 키는 *_scaled 컬럼명 사용 (예: brand_loyalty_scaled)
BEHAV_FIELD_MULTIPLIERS = {
    # 예: "brand_loyalty_scaled": 1.2,
    #     "price_sensitivity_scaled": 0.9,
}

print("CONFIG loaded. Adjust DEMO_SHARE / BEHAV_SHARE / field weights as needed.")


In [ ]:

# =============================
# 1) Load CSV (encoding fallback) & column name cleaning
# =============================
import pandas as pd
import re

def read_csv_fallback(path: Path):
    try:
        return pd.read_csv(path)
    except UnicodeDecodeError:
        return pd.read_csv(path, encoding="cp949")

df = read_csv_fallback(INPUT_CSV)

# 일부 컬럼명이 'brand_loyalty'처럼 따옴표를 포함할 수 있어 정리
def clean_colname(c: str) -> str:
    return re.sub(r"'([^']+)'", r"\1", c)

df.rename(columns={c: clean_colname(c) for c in df.columns}, inplace=True)

print("Rows:", len(df))
print("Head columns:", df.columns[:15].tolist())
df.head(2)


In [ ]:

# =============================
# 2) Identify demographics & behavioral columns
# =============================

# 프로젝트 기준 인구통계 후보 (파일에 따라 존재 여부 다를 수 있음)
candidate_demo = ["SQ3", "HQ2", "SQ4", "SQ5", "SQ8", "DQ1", "DQ3", "DQ3_1"]
demo_cols = [c for c in candidate_demo if c in df.columns]

# *_scaled 로 끝나는 세그먼트/성향 점수
behav_cols = [c for c in df.columns if c.endswith("_scaled")]

if not behav_cols:
    raise RuntimeError("No behavioral (_scaled) columns detected. Please check the CSV headers.")

print("Demographics columns:", demo_cols)
print("Behavioral columns:", behav_cols)


In [ ]:

# =============================
# 3) Helpers
# =============================
from typing import Dict, Any

def normalize_weights(raw: Dict[str, float], target_sum: float) -> Dict[str, float]:
    """raw 딕셔너리를 합=target_sum 이 되도록 정규화. raw 합이 0이면 균등 분배."""
    total = sum(v for v in raw.values() if pd.notna(v))
    if total <= 0:
        n = len(raw)
        return {k: (target_sum / n if n else 0.0) for k in raw}
    return {k: (v / total) * target_sum for k, v in raw.items()}

def build_persona_attributes(row: pd.Series,
                             demo_cols, behav_cols,
                             demo_share: float, behav_share: float,
                             demo_field_weights: Dict[str, float],
                             behav_field_multipliers: Dict[str, float]) -> Dict[str, Any]:
    # 3a) Demographics — 내부 가중 지정 없으면 균등(=1.0)
    demo_raw = {c: demo_field_weights.get(c, 1.0) for c in demo_cols}
    demo_weights = normalize_weights(demo_raw, demo_share)
    demo_attrs = {
        c: {
            "value": (None if pd.isna(row.get(c)) else row.get(c)),
            "weight": float(demo_weights.get(c, 0.0))
        }
        for c in demo_cols
    }

    # 3b) Behavioral — scaled 값 × multiplier
    behav_raw = {}
    for c in behav_cols:
        val = row.get(c)
        if pd.isna(val):
            val = 0.0
        mult = behav_field_multipliers.get(c, 1.0)
        behav_raw[c] = float(val) * float(mult)

    behav_weights = normalize_weights(behav_raw, behav_share)
    behav_attrs = {
        c: {
            "value": float(row.get(c, 0.0)),
            "weight": float(behav_weights.get(c, 0.0))
        }
        for c in behav_cols
    }

    # Merge & final tiny rescale to make sum ~ 1.0 exactly
    attributes = {**demo_attrs, **behav_attrs}
    total_weight = sum(v["weight"] for v in attributes.values())
    if total_weight > 0:
        for k in attributes:
            attributes[k]["weight"] = attributes[k]["weight"] / total_weight

    return attributes


In [ ]:

# =============================
# 4) Build weighted attributes & save JSONL
# =============================
import json

# 안전장치: DEMO_SHARE + BEHAV_SHARE 자동 정규화(권장 X, 경고만)
total_share = DEMO_SHARE + BEHAV_SHARE
if abs(total_share - 1.0) > 1e-8:
    print(f"[WARN] DEMO_SHARE + BEHAV_SHARE = {total_share:.4f} (≠ 1.0). 자동 정규화합니다.")
    DEMO_SHARE = DEMO_SHARE / total_share
    BEHAV_SHARE = BEHAV_SHARE / total_share
    print(f" -> DEMO_SHARE={DEMO_SHARE:.4f}, BEHAV_SHARE={BEHAV_SHARE:.4f}")

records = []
for idx, row in df.iterrows():
    persona = {
        "persona_key": row.get("id", f"row_{idx}"),
        "attributes": build_persona_attributes(row,
                                              demo_cols, behav_cols,
                                              DEMO_SHARE, BEHAV_SHARE,
                                              DEMO_FIELD_WEIGHTS,
                                              BEHAV_FIELD_MULTIPLIERS)
    }
    records.append(persona)

# Save JSONL
with OUT_JSONL.open("w", encoding="utf-8") as f:
    for rec in records:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")

print(f"Saved JSONL -> {OUT_JSONL}")
print("Total personas:", len(records))


In [ ]:

# =============================
# 5) Preview (first 3)
# =============================
import json

preview = records[:3]
with OUT_PREVIEW.open("w", encoding="utf-8") as f:
    f.write(json.dumps(preview, ensure_ascii=False, indent=2))

print(f"Saved preview -> {OUT_PREVIEW}")
preview
